# Fine-Tuning Local LLM: Penerjemah Slang → Bahasa Indonesia Baku
Model: **Qwen 2.5 1.5B Instruct** | Metode: **QLoRA (Unsloth)** | Ekspor: **GGUF q4_k_m untuk Ollama**

**Sebelum mulai**
1. Menu *Runtime → Change runtime type → T4 GPU*.
2. Siapkan file `dataset_slang.jsonl` (akan diminta upload di langkah 2).
3. Jalankan sel dari atas ke bawah, satu per satu.

**Alur:** install → upload & analisis dataset → load model → uji model dasar (SEBELUM) → training → kurva loss → uji model hasil fine-tuning (SESUDAH) → ekspor GGUF.

## 1. Instalasi

In [ ]:
%%capture
!pip install unsloth sacrebleu

## 2. Upload & Analisis Dataset

In [ ]:
from google.colab import files
uploaded = files.upload()   # pilih dataset_slang.jsonl
DATA_FILE = list(uploaded.keys())[0]
print("File:", DATA_FILE)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_json(DATA_FILE, lines=True)
print("Jumlah data awal:", len(df))

# Preprocessing: rapikan spasi & hapus duplikat berdasarkan input
for c in ["instruction", "input", "output"]:
    df[c] = df[c].astype(str).str.strip()
df = df.drop_duplicates(subset=["input"]).reset_index(drop=True)
print("Jumlah data setelah hapus duplikat:", len(df))

# Analisis singkat data (untuk laporan)
df["kata_input"]  = df["input"].str.split().str.len()
df["kata_output"] = df["output"].str.split().str.len()
ringkasan = df.groupby("kategori").agg(
    jumlah=("input", "count"),
    rata_kata_input=("kata_input", "mean"),
    rata_kata_output=("kata_output", "mean"),
).round(1)
display(ringkasan)

ringkasan["jumlah"].plot(kind="bar", title="Distribusi data per kategori", rot=30)
plt.ylabel("jumlah pasang"); plt.tight_layout()
plt.savefig("distribusi_data.png", dpi=150); plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

train_df, val_df = train_test_split(df, test_size=0.1, random_state=42, stratify=df["kategori"])
print("Train:", len(train_df), "| Validation:", len(val_df))
train_ds = Dataset.from_pandas(train_df[["instruction", "input", "output"]].reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df[["instruction", "input", "output"]].reset_index(drop=True))

## 3. Load Model Dasar (4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ = 512
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    dtype=None,
)

## 4. Format Prompt & Fungsi Terjemah

In [ ]:
SYSTEM_PROMPT = ("Kamu adalah penerjemah yang mengubah bahasa gaul (slang) "
                 "menjadi bahasa Indonesia baku. Jawab hanya dengan hasil terjemahan.")
INS_SLANG  = "Ubah kalimat slang berikut menjadi bahasa Indonesia baku."

def to_messages(ins, inp, out=None):
    m = [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": f"{ins}\n\n{inp}"}]
    if out is not None:
        m.append({"role": "assistant", "content": out})
    return m

def fmt(ex):
    return {"text": tokenizer.apply_chat_template(
        to_messages(ex["instruction"], ex["input"], ex["output"]), tokenize=False)}

def translate(m, ins, inp, max_new_tokens=96):
    prompt = tokenizer.apply_chat_template(to_messages(ins, inp), tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, use_cache=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Kalimat uji DI LUAR dataset latih (untuk demo sebelum vs sesudah)
TEST = [
 (INS_SLANG,  "anjir, tugas gue numpuk banget minggu ini"),
 (INS_SLANG,  "woi, jangan lupa bawain charger gue ya"),
 (INS_SLANG,  "doi udh balikan sm mantannya, bkn gue yg salah"),
 (INS_SLANG,  "gila, nilai gue naik drastis semester ini"),
 (INS_SLANG,  "sorry bgt gue telat, tadi ban motor gue bocor"),
 (INS_SLANG,  "lo mau titip apa? gue mau ke indomaret"),
 (INS_SLANG,  "gue males banget deh ngerjain laporan praktikum"),
 (INS_SLANG,  "btw besok kelasnya jadi online apa offline sih?"),
 (INS_SLANG,  "ntar malem gue nginep di rumah temen"),
 (INS_SLANG,  "dosennya galak banget, gue jadi gugup pas presentasi"),
 (INS_SLANG,  "capek banget gue abis nugas semaleman"),
 (INS_SLANG,  "eh lo udah makan belum? gue laper nih"),
 (INS_SLANG,  "bokap gue marah gara-gara gue pulang kemaleman"),
 (INS_SLANG,  "tugas kelompok kita bubar jalan, nggak ada yang mau ngerjain"),
 (INS_SLANG,  "gue sih oke aja, terserah lo deh"),
]

## 5. Uji Model DASAR (SEBELUM fine-tuning)

In [ ]:
import sacrebleu

N_VAL = min(30, len(val_ds))
base_test = [translate(model, i, t) for i, t in TEST]
base_val  = [translate(model, val_ds[k]["instruction"], val_ds[k]["input"]) for k in range(N_VAL)]
refs_val  = [val_ds[k]["output"] for k in range(N_VAL)]

for (i, t), o in zip(TEST, base_test):
    print("INPUT :", t); print("BASE  :", o); print("-"*60)

print("chrF model dasar (validation):", round(sacrebleu.corpus_chrf(base_val, [refs_val]).score, 2))

## Konfigurasi eksperimen
Ubah nilai di bawah untuk membandingkan konfigurasi. Contoh untuk laporan: jalankan notebook sekali dengan **R=16, EPOCHS=5** (`TAG="r16_e5"`), lalu **R=16, EPOCHS=2** (`TAG="r16_e2"`) dan bandingkan skor validation-nya. Nama file hasil otomatis memakai TAG, jadi tidak saling menimpa.

In [ ]:
LORA_R = 16
LORA_ALPHA = 16      # umumnya sama dengan R (atau 2x R)
EPOCHS = 2           # percobaan pertama (5 epoch) overfit: validation loss terendah di sekitar epoch 1.5-2
TAG = "r16_e2"       # label konfigurasi, dipakai pada nama file hasil

## 6. Pasang LoRA (QLoRA)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,             # rank LoRA
    lora_alpha=LORA_ALPHA, # alpha
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
model.print_trainable_parameters()

## 7. Training

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

train_fmt = train_ds.map(fmt)
val_fmt   = val_ds.map(fmt)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_fmt,
    eval_dataset=val_fmt,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=EPOCHS,
        learning_rate=2e-4,
        lr_scheduler_type="linear",
        warmup_steps=5,
        weight_decay=0.01,
        optim="adamw_8bit",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=5,
        eval_strategy="steps",
        eval_steps=10,
        save_strategy="no",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
# Loss hanya dihitung pada bagian jawaban (terjemahan), bukan pada prompt
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
stats = trainer.train()
print(stats.metrics)

## 8. Kurva Loss & Log Training

In [ ]:
import json
logs = trainer.state.log_history
json.dump(logs, open(f"log_training_{TAG}.json", "w"), indent=2)

tr = [(l["step"], l["loss"]) for l in logs if "loss" in l]
ev = [(l["step"], l["eval_loss"]) for l in logs if "eval_loss" in l]
plt.figure(figsize=(7,4))
plt.plot(*zip(*tr), label="train loss")
if ev: plt.plot(*zip(*ev), marker="o", label="validation loss")
plt.xlabel("step"); plt.ylabel("loss"); plt.title("Kurva loss fine-tuning"); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"kurva_loss_{TAG}.png", dpi=150); plt.show()

## 9. Uji Model Fine-Tuned (SESUDAH) & Perbandingan

In [ ]:
FastLanguageModel.for_inference(model)

ft_test = [translate(model, i, t) for i, t in TEST]
ft_val  = [translate(model, val_ds[k]["instruction"], val_ds[k]["input"]) for k in range(N_VAL)]

hasil = pd.DataFrame({
    "input": [t for _, t in TEST],
    "model_dasar": base_test,
    "model_finetuned": ft_test,
})
hasil.to_csv(f"hasil_perbandingan_{TAG}.csv", index=False)
pd.set_option("display.max_colwidth", 80)
display(hasil)

c_base = sacrebleu.corpus_chrf(base_val, [refs_val]).score
c_ft   = sacrebleu.corpus_chrf(ft_val,   [refs_val]).score
print(f"chrF validation -> dasar: {c_base:.2f} | fine-tuned: {c_ft:.2f}")

## 10. Simpan Adapter & Ekspor ke GGUF

In [ ]:
model.save_pretrained("lora_slang_baku")
tokenizer.save_pretrained("lora_slang_baku")

# Ekspor GGUF terkuantisasi 4-bit (untuk Ollama). Butuh beberapa menit.
model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method="q4_k_m")
# Unsloth versi baru bisa menyimpan hasil ke folder "gguf_model_gguf". Cari semua file .gguf:
import glob, os
for p in glob.glob("**/*.gguf", recursive=True):
    print(p, round(os.path.getsize(p)/1e6), "MB")

In [ ]:
# Simpan semua hasil ke Google Drive (disarankan, file GGUF cukup besar)
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/proyek_llm_slang
# Salin HANYA file .gguf (bukan seluruh folder, karena folder berisi bobot 16-bit yang besar)
import shutil
D = "/content/drive/MyDrive/proyek_llm_slang"
os.makedirs(D, exist_ok=True)
ggufs = glob.glob("**/*.gguf", recursive=True)
print("File GGUF ditemukan:", ggufs)
shutil.copy(ggufs[0], f"{D}/model_{TAG}.Q4_K_M.gguf")
shutil.copytree("lora_slang_baku", f"{D}/lora_{TAG}", dirs_exist_ok=True)
for f in [f"kurva_loss_{TAG}.png", "distribusi_data.png", f"log_training_{TAG}.json", f"hasil_perbandingan_{TAG}.csv"]:
    shutil.copy(f, D)
print(os.listdir(D))

## 11. Modelfile untuk Ollama (jalankan di laptop)
Salin file `.gguf` dari Drive ke satu folder di laptop, buat file bernama `Modelfile` dengan isi di bawah
(ganti nama file `.gguf` sesuai hasil `ls` di atas), lalu jalankan:

```
ollama create slang-baku -f Modelfile
ollama run slang-baku
```

**Modelfile:**
```
FROM ./unsloth.Q4_K_M.gguf
TEMPLATE """{{- if .System }}<|im_start|>system
{{ .System }}<|im_end|>
{{ end }}<|im_start|>user
{{ .Prompt }}<|im_end|>
<|im_start|>assistant
"""
SYSTEM """Kamu adalah penerjemah yang mengubah bahasa gaul (slang) menjadi bahasa Indonesia baku. Jawab hanya dengan hasil terjemahan."""
PARAMETER stop "<|im_end|>"
PARAMETER temperature 0.2
```

Format input di Ollama harus sama dengan saat training, yaitu:
`Ubah kalimat slang berikut menjadi bahasa Indonesia baku.` + baris kosong + kalimatnya.
(UI Gradio nanti yang menyusun format ini otomatis.)

## Troubleshooting (catat error yang kamu temui untuk laporan)
| Error | Penyebab umum | Solusi |
|---|---|---|
| `CUDA out of memory` | batch terlalu besar | turunkan `per_device_train_batch_size` jadi 2, naikkan `gradient_accumulation_steps` |
| `TypeError ... max_seq_length` pada SFTConfig | versi TRL baru | ganti `max_seq_length` menjadi `max_length` |
| `unexpected keyword argument 'tokenizer'` | versi TRL baru | ganti `tokenizer=tokenizer` menjadi `processing_class=tokenizer` |
| Ekspor GGUF gagal / lama | build llama.cpp di Colab | jalankan ulang sel; pastikan runtime GPU aktif |
| Output model berulang / ngawur | data terlalu sedikit atau epoch berlebih | tambah data, kurangi epoch, bandingkan validation loss |